## Evaluation Framework

In order to set up an evaluation, we need to consider what we are aiming to get. Down the line, the ESCO nodes found by our vector search will be used to build a prompt so that the Large Language Model can ask users to validate a number of skills and occupations found through vector search. For that reason, we don't want to focus so much on the precision (that is, reducing the amount of false positives), but rather on the recall (that is, reducing the amount of false negatives). 

On this premise, we design an evaluation method (accuracy@k) in which we consider a success (a score of 1) if the correct node is found within the top k classes and an insuccess (a score of 0) otherwise. This method could be further refined by considering the score to be inversely proportional to the rank in which the correct node is found. However, since we're not concerned with the rank, we will use an averaged 0-1 score.

In [4]:
from typing import List

def accuracy_within_list(prediction: List[List[str]], true: List[str]):
    """Calculates the average accuracy considering
    for each prediction a value of 1 if the correct
    node is in the predicted list and 0 otherwise.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[str]): list of the true nodes in the
            dataset.

    Returns:
        float: average accuracy over all the test set.
    """
    assert len(prediction) == len(true)
    total_correct = 0
    for pred_list, true_val in zip(prediction, true):
        total_correct+=int(true_val in pred_list)
    return total_correct/len(true)

def accuracy_at_k(prediction: List[List[str]], true: List[str], k: int):
    """Calculates the average accuracy considering
    for each prediction a value of 1 if the correct
    node is in the top k elements of the predicted list
    and 0 otherwise.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[str]): list of the true nodes in the
            dataset.
        k (int): number of elements of each list to be considered.

    Returns:
        float: average accuracy over all the test set.
    """
    assert len(prediction) == len(true)
    total_correct = 0
    for pred_list, true_val in zip(prediction, true):
        total_correct+=int(true_val in pred_list[:k])
    return total_correct/len(true)

## Evaluation goals

The objective of the evaluation is to choose which model and hyperparameters have the highest accuracy at k. For a given embedding model, the hyperparameters are the following:

1. **How to embed a node of the graph**: which combination of the fields guarantees the best performance when embedded.
2. **Score function**: which function should be used to retrieve the most similar nodes (cosine, l2 distance, etc.)
3. **Using Maximal Marginal Relevance**: whether we should use MMR to get more diverse results.

We will use ChromaDB as a local vector store and save our data on the csv file.

In [5]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv("/Users/francescopreta/coding/compass/backend/.env")

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
REPO_ID = "tabiya/hahu_test"
FILENAME = "redacted_hahu_test_with_id.csv"

df_test = pd.read_csv(
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset", token=HF_TOKEN)
)

OCCUPATION_CSV_PATH = "/Users/francescopreta/coding/compass/poc/data/occupations.csv"
df_database = pd.read_csv(OCCUPATION_CSV_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

/Users/francescopreta/miniconda3/envs/backend/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# We pre-compute the embeddings for each method
def description(df):
    return df["DESCRIPTION"]

def preferred_label(df):
    return df["PREFERREDLABEL"]

def all_occupations(df):
    return f"""Occupation Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Occupation Description: {df['DESCRIPTION']}"""

def label_and_description(df):
    return f"{df['PREFERREDLABEL']}\n{df['DESCRIPTION']}"

function_to_method ={"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_OCCUPATIONS":all_occupations, "LABEL_AND_DESCRIPTION": label_and_description}
num_samples = len(df_database)
batch_size = 100
for method in function_to_method:
    df_database[method] = df_database.progress_apply(function_to_method[method], axis=1)
    embedding_results = []
    all_strings = list(df_database[method])
    for i in range(int(num_samples/batch_size)):
        batch = all_strings[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    batch = all_strings[(i+1)*batch_size:]
    embedding_results += model.get_embeddings(batch)
    df_database[f"embeddings_{method}"] = [embedding_results[i].values for i in range(len(df_database))]


  0%|          | 0/3007 [00:00<?, ?it/s]

100%|██████████| 3007/3007 [00:00<00:00, 258225.96it/s]


3007


100%|██████████| 3007/3007 [00:00<00:00, 403579.79it/s]


3007


100%|██████████| 3007/3007 [00:00<00:00, 225191.00it/s]


3007


100%|██████████| 3007/3007 [00:00<00:00, 266480.85it/s]


3007


In [22]:
import chromadb

def db_vector_search(query: str, embedding_model: TextEmbeddingModel, collection: chromadb.Collection):
    embeddings_raw = embedding_model.get_embeddings([query])
    embeddings = embeddings_raw[0].values
    return collection.query(query_embeddings=embeddings, n_results=10)


In [17]:
import chromadb

embedding_methods = ["DESCRIPTION", "PREFERREDLABEL", "ALL_OCCUPATIONS", "LABEL_AND_DESCRIPTION"]
score_function = ["cosine", "l2", "ip"]
mmr = [False, True]
K=10

for method in embedding_methods:
    for score in score_function:
        collection_name = f'{method}__{score}'
        collection_metadata = {"hnsw:space":score}
        client = chromadb.Client()
        collection = client.create_collection(name=collection_name, metadata=collection_metadata)
        collection.add(
            documents = list(df_database[method]),
            # check if you can add a dataframe
            metadatas = [{"esco_code":row["CODE"]} for _, row in df_database.iterrows()],
            embeddings = list(df_database[f"embeddings_{method}"]),
            ids = [f"id_{i}" for i in range(len(df_database))]
        )
        
        for m in mmr:
            if m:
                print(collection.query("This query is about fishing", K))
                df_test[collection_name] = df_test["synthetic_query"].progress_apply(lambda x: [elem.esco_code for elem in db.max_marginal_relevance_search(x, K)])
            else:
                df_test[collection_name] = df_test["synthetic_query"].progress_apply(lambda x: [elem.esco_code for elem in db.similarity_search(x, K)])
            for k in [1, 3, 5, 10]:
                acc_at_k = accuracy_at_k(list(df_test[collection_name]), list(df_test["esco_code"]))
                print(f"Method: {method}, Score function: {score}, MMR: {m}\n Accuracy at {k}: {round(acc_at_k,4)}")

            



  0%|          | 1/550 [00:00<00:00, 11915.64it/s]


NameError: name 'db' is not defined

In [23]:
db_vector_search("I like fishing", model, collection)

{'ids': [['id_2245',
   'id_3001',
   'id_83',
   'id_1624',
   'id_2279',
   'id_2368',
   'id_455',
   'id_733',
   'id_1377',
   'id_308']],
 'distances': [[0.30110806226730347,
   0.31396472454071045,
   0.3157963156700134,
   0.317432165145874,
   0.3262636661529541,
   0.33304470777511597,
   0.33392924070358276,
   0.340063214302063,
   0.34050971269607544,
   0.3405410051345825]],
 'metadatas': [[{'esco_code': '5120.1.2'},
   {'esco_code': '6223.2.1'},
   {'esco_code': '5223.7.16'},
   {'esco_code': '8160.32'},
   {'esco_code': '6223.2'},
   {'esco_code': '3359.3'},
   {'esco_code': '6223.1'},
   {'esco_code': '2132.4'},
   {'esco_code': '8160.31'},
   {'esco_code': '8350.4'}]],
 'embeddings': None,
 'documents': [['Fish cooks are responsible for preparing and presenting fish dishes using a variety of techniques. They may also prepare the accompanying sauces and purchase fresh fish for these dishes.',
   'Fisheries boatmasters operate fishing vessels in coastal waters performin